# Tahap 1 — Membangun Case Base

Tahap ini mencakup:
1. **Konversi PDF ke Plain Text** menggunakan `pdfminer.six`
2. **Cleaning & Normalisasi** teks putusan

**Input**: PDF putusan di `data/raw/pasal_362/` dan `data/raw/pasal_363/`

**Output**: Teks bersih di `data/processed/clean/pasal_362/` dan `data/processed/clean/pasal_363/`

> **Catatan**: Proses pengumpulan data (scraping dari Direktori Mahkamah Agung RI) sudah dilakukan secara terpisah menggunakan `scraper_pn_malang.py`. Total 60 putusan (30 Pasal 362 + 30 Pasal 363) sudah tersedia di folder `data/raw/`.

## 1.1 Import Library

In [1]:
import re
import logging
from pathlib import Path
from tqdm.notebook import tqdm
from pdfminer.high_level import extract_text
from pdfminer.pdfparser import PDFSyntaxError

## 1.2 Konfigurasi

In [2]:
RAW_DIR   = Path("../data/raw")
CLEAN_DIR = Path("../data/processed/clean")
LOG_DIR   = Path("../logs")
MIN_CHARS = 500

for d in [CLEAN_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "cleaning.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

pdf_files = sorted(RAW_DIR.rglob("*.pdf"))
print(f"Total PDF ditemukan: {len(pdf_files)}")
for folder in ["pasal_362", "pasal_363"]:
    count = len(list((RAW_DIR / folder).glob("*.pdf")))
    print(f"  {folder}: {count} file")

Total PDF ditemukan: 60
  pasal_362: 30 file
  pasal_363: 30 file


## 1.3 Konversi PDF ke Plain Text

Mengubah file PDF menjadi plain text menggunakan `pdfminer.six`. Proses ini di-skip otomatis jika file `.txt` sudah ada.

In [3]:
def pdf_to_text(pdf_path):
    try:
        text = extract_text(str(pdf_path))
        if not text or len(text.strip()) < MIN_CHARS:
            log.warning(f"Teks terlalu pendek: {pdf_path.name}")
            return None
        return text
    except PDFSyntaxError as e:
        log.error(f"PDF corrupt: {pdf_path.name} — {e}")
        return None
    except Exception as e:
        log.error(f"Error: {pdf_path.name}: {e}")
        return None


success = failed = skipped = 0

for pdf_path in tqdm(pdf_files, desc="Konversi PDF ke TXT"):
    txt_path = pdf_path.with_suffix(".txt")
    if txt_path.exists():
        skipped += 1
        continue
    text = pdf_to_text(pdf_path)
    if text is None:
        failed += 1
        continue
    txt_path.write_text(text, encoding="utf-8")
    success += 1

print(f"\nBerhasil: {success} | Gagal: {failed} | Skip (sudah ada): {skipped}")

Konversi PDF ke TXT:   0%|          | 0/60 [00:00<?, ?it/s]


Berhasil: 0 | Gagal: 0 | Skip (sudah ada): 60


## 1.4 Cleaning & Normalisasi Teks

Pipeline cleaning:
1. Hapus header/footer berulang (watermark MA, nomor halaman)
2. Hapus karakter non-latin dan karakter tidak diperlukan
3. Normalisasi spasi dan baris kosong
4. Lowercase
5. Validasi minimal 100 kata

In [4]:
def clean_text(text):
    patterns_hapus = [
        r"Direktori Putusan Mahkamah Agung Republik Indonesia",
        r"putusan\.mahkamahagung\.go\.id",
        r"Disclaimer\s*[::].*",
        r"kepaniteraan mahkamah agung.*",
        r"mahkamah agung republik indonesia",
        r"halaman \d+ dari \d+",
        r"hal\.\s*\d+\s*dari\s*\d+",
        r"-{3,}",
        r"={3,}",
        r"\f",
    ]
    for pat in patterns_hapus:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)
    text = text.replace("\xa0", " ").replace("\t", " ")
    text = re.sub(r"[^\x00-\x7F\u00C0-\u024F\u1E00-\u1EFF]", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip().lower()
    return text


def validate_text(text, filename):
    word_count = len(text.split())
    if word_count < 100:
        log.warning(f"VALIDASI GAGAL — terlalu pendek ({word_count} kata): {filename}")
        return False
    return True


txt_files = sorted(RAW_DIR.rglob("*.txt"))
print(f"Total TXT ditemukan: {len(txt_files)}")

success = failed = skipped = 0

for txt_path in tqdm(txt_files, desc="Cleaning & Normalisasi"):
    label_folder = txt_path.parent.name
    out_dir = CLEAN_DIR / label_folder
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / txt_path.name

    if out_path.exists():
        skipped += 1
        continue

    try:
        raw_text = txt_path.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        log.error(f"Gagal baca {txt_path.name}: {e}")
        failed += 1
        continue

    clean = clean_text(raw_text)
    if not validate_text(clean, txt_path.name):
        failed += 1
        continue

    out_path.write_text(clean, encoding="utf-8")
    success += 1

print(f"\nBerhasil: {success} | Gagal: {failed} | Skip (sudah ada): {skipped}")
print(f"Output disimpan di: {CLEAN_DIR.resolve()}")

Total TXT ditemukan: 60


Cleaning & Normalisasi:   0%|          | 0/60 [00:00<?, ?it/s]


Berhasil: 0 | Gagal: 0 | Skip (sudah ada): 60
Output disimpan di: C:\Users\LENOVO\OneDrive\PRAKTIKUM\penalaran komputer\data\processed\clean


## 1.5 Validasi Hasil

In [5]:
clean_files = sorted(CLEAN_DIR.rglob("*.txt"))
print(f"Total file clean: {len(clean_files)}")

for folder in ["pasal_362", "pasal_363"]:
    count = len(list((CLEAN_DIR / folder).glob("*.txt")))
    print(f"  {folder}: {count} file")

if clean_files:
    sample = clean_files[0]
    text = sample.read_text(encoding="utf-8")
    print(f"\nSample ({sample.name}):")
    print(text[:500])
    print(f"\n... ({len(text.split())} kata total)")

Total file clean: 60
  pasal_362: 30 file
  pasal_363: 30 file

Sample (case_pasal_362_001.txt):
disclaimer
 
pelaksanaan fungsi peradilan. namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.
dalam hal anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi 
email : kepaniteraan@mahkamahagung.go.id telp : 021-384 3348 (ext.318)

halaman 1

putusannomor 132/p

... (4500 kata total)
